# Linear diffusion: FTCS vs PINN

**Book:** §3.5, Figure 3.3(a) &nbsp;·&nbsp; `ch03/diffusion_ftcs_vs_pinn.ipynb`

$$u_t = \alpha\,u_{xx},\qquad u(0,t)=u(1,t)=0,\qquad u(x,0)=\sin\pi x + 0.3\sin 3\pi x,$$

$\alpha=0.2$, exact $u = e^{-\alpha\pi^2 t}\sin\pi x + 0.3\,e^{-9\alpha\pi^2 t}\sin 3\pi x$.
Each mode decays $\propto k^2$: the third harmonic dies **nine times faster**. Diffusion is a
low-pass filter.

**CFD.** Explicit FTCS, $u_i^{n+1}=u_i^n+\frac{\alpha\Delta t}{\Delta x^2}(u_{i+1}^n-2u_i^n+u_{i-1}^n)$,
stable only for $\alpha\Delta t/\Delta x^2\le\frac12$ — the punishing $\Delta t\sim\Delta x^2$.

**PINN loss.**
$$\mathcal L = \overline{(u_t-\alpha u_{xx})^2} + 20\,\overline{(u(x,0)-u_0(x))^2} + 20\,\overline{u(0,t)^2 + u(1,t)^2}.$$

**Verdict.** The classical method wins on both counts here — expect it to.

In [ ]:
import time
import numpy as np, torch, torch.nn as nn
import matplotlib.pyplot as plt
torch.manual_seed(0); np.random.seed(0)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
def g1(f, x): return torch.autograd.grad(f, x, torch.ones_like(f), create_graph=True)[0]

AL, TMAX = 0.2, 0.5
u0    = lambda x: np.sin(np.pi*x) + 0.3*np.sin(3*np.pi*x)
uex   = lambda x, t: np.exp(-AL*np.pi**2*t)*np.sin(np.pi*x) \
                   + 0.3*np.exp(-9*AL*np.pi**2*t)*np.sin(3*np.pi*x)

# ---------------- CFD: explicit FTCS ----------------
def ftcs(nx=201):
    x  = np.linspace(0, 1, nx); dx = x[1]-x[0]
    dt = 0.4*dx**2/AL                      # safely inside the r <= 1/2 limit
    nt = int(np.ceil(TMAX/dt)); dt = TMAX/nt
    r  = AL*dt/dx**2
    u  = u0(x)
    t0 = time.perf_counter()
    for _ in range(nt):
        u[1:-1] = u[1:-1] + r*(u[2:] - 2*u[1:-1] + u[:-2])
        u[0] = u[-1] = 0.0
    return x, u, time.perf_counter()-t0, nt, r

xg, u_fd, t_fd, nt, r = ftcs()
u_ex = uex(xg, TMAX)
e_fd = np.sqrt(np.mean((u_fd-u_ex)**2)/np.mean(u_ex**2))
print(f'FTCS  : {nt} steps (r={r:.2f})   {t_fd*1e3:.0f} ms   rel L2 = {e_fd:.1e}')

# ---------------- PINN ----------------
net = nn.Sequential(nn.Linear(2,48), nn.Tanh(), nn.Linear(48,48), nn.Tanh(),
                    nn.Linear(48,48), nn.Tanh(), nn.Linear(48,1)).to(device)
opt = torch.optim.Adam(net.parameters(), 2e-3)
xi  = torch.linspace(0,1,200,device=device).reshape(-1,1)
ui  = torch.tensor(u0(xi.cpu().numpy()), dtype=torch.float32, device=device)
t0 = time.perf_counter()
for e in range(8000):
    if e == 6000:
        for g in opt.param_groups: g['lr'] = 4e-4
    opt.zero_grad()
    x = torch.rand(2000,1,device=device).requires_grad_(True)
    t = (torch.rand(2000,1,device=device)*TMAX).requires_grad_(True)
    u = net(torch.cat([x,t],1))
    res = g1(u,t) - AL*g1(g1(u,x),x)
    tb  = torch.rand(200,1,device=device)*TMAX
    z, o = torch.zeros_like(tb), torch.ones_like(tb)
    loss = (res**2).mean() \
         + 20*((net(torch.cat([xi, torch.zeros_like(xi)],1)) - ui)**2).mean() \
         + 20*(net(torch.cat([z,  tb],1))**2).mean() \
         + 20*(net(torch.cat([o,  tb],1))**2).mean()
    loss.backward(); opt.step()
if device.type=='cuda': torch.cuda.synchronize()
t_pinn = time.perf_counter()-t0

xt = torch.tensor(xg, dtype=torch.float32, device=device).reshape(-1,1)
with torch.no_grad():
    u_pn = net(torch.cat([xt, torch.full_like(xt, TMAX)],1)).cpu().numpy().ravel()
e_pn = np.sqrt(np.mean((u_pn-u_ex)**2)/np.mean(u_ex**2))
print(f'PINN  : {t_pinn:.0f} s   rel L2 = {e_pn:.1e}')
print(f'\nCFD is {t_pinn/t_fd:.0f}x faster and {e_pn/e_fd:.0f}x more accurate. Use the CFD.')

plt.figure(figsize=(8,4.4))
plt.plot(xg, u0(xg),  'k:',  lw=1.2, label='initial profile')
plt.plot(xg, u_ex,    'g',   lw=2.6, alpha=.6, label=f'exact, t={TMAX}')
plt.plot(xg, u_fd,    'b--', lw=1.5, label=f'FTCS ({e_fd:.1e}, {t_fd*1e3:.0f} ms)')
plt.plot(xg, u_pn,    'r--', lw=1.5, label=f'PINN ({e_pn:.1e}, {t_pinn:.0f} s)')
plt.xlabel('x'); plt.ylabel('u'); plt.legend(fontsize=9); plt.grid(alpha=.3)
plt.title('Linear diffusion: the third harmonic has all but vanished')
plt.tight_layout(); plt.show()